# GraphRAG Local Search and source evidence

Local Search combines graph information with source text to build a question-specific
context. The native model prompt generates the response from that mixed context.
For common retrieval evaluation, only the ordered Sources list is projected onto
the original page-based evidence units.

See the [GraphRAG workflow](../docs/illustrations/graphrag-workflow.pdf)
for indexing, Local Search and source-page evaluation.

Read [the index notebook](03_graphrag_index.ipynb) first. Use the GraphRAG extra
described in the README. Default execution reads saved RFC 2795 results without
credentials or model requests. New queries use Azure only when explicitly selected.
See thesis Section 2.5 and Appendix G for details.

In [1]:
import hashlib
import json
import math
import importlib.metadata
import os
from contextlib import chdir
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import verify_artifacts
from rag_comparison.graphrag_projection import (
    build_text_unit_index,
    project_local_search_context,
)
from rag_comparison.metrics import ndcg_at_k

## Select a stored index and query mode

The index manifest supplies native tables, questions and the projection ledger.
Set `RUN_AZURE_QUERY = True` only for a new query run, using an ignored index folder
prepared by the preceding notebook. Existing responses are reused when their index
binding matches. The default mode reads the saved public contexts and responses.

For private results, select the approved run folder in an ignored notebook copy. Set `reference_path` to its matching reference package, or to `None` to inspect contexts and projections without calculating a metric.

In [2]:
RUN_AZURE_QUERY = False
azure_dir = Path("../local/graphrag/rfc2795-azure-example")
reference_path = Path("../examples/data/reference_packages.json")
data_dir = azure_dir if RUN_AZURE_QUERY else Path("../examples/graphrag")
index_manifest = json.loads(
    (data_dir / "index_manifest.json").read_text(encoding="utf-8")
)
assert index_manifest["graphrag_version"] == config.GRAPHRAG_VERSION
assert index_manifest["status"] == "completed"
assert importlib.metadata.version("graphrag") == config.GRAPHRAG_VERSION
condition = config.CONDITIONS[index_manifest["condition_id"]]
assert condition["system"] == "G"
embedding_variable = config.EMBEDDING_DEPLOYMENT_VARIABLES[condition["embedding_model"]]
paths = verify_artifacts(data_dir, index_manifest)
questions = [
    json.loads(line)
    for line in paths["questions"].read_text(encoding="utf-8").splitlines()
    if line
]
question_texts = {row["question_id"]: row["question_text"] for row in questions}
assert len(question_texts) == len(questions)
index_sha = hashlib.sha256((data_dir / "index_manifest.json").read_bytes()).hexdigest()
query_parameters = {
    "community_level": config.GRAPHRAG_COMMUNITY_LEVEL,
    "response_type": config.GRAPHRAG_RESPONSE_TYPE,
}
tables = {
    name: pd.read_parquet(paths[name])
    for name in (
        "documents",
        "text_units",
        "entities",
        "relationships",
        "communities",
        "community_reports",
    )
}
ledger = [
    json.loads(line)
    for line in paths["input_projection"].read_text(encoding="utf-8").splitlines()
    if line
]
text_unit_index = build_text_unit_index(
    tables["text_units"]["human_readable_id"].tolist(),
    tables["text_units"]["document_id"].tolist(),
)
projection_edges = {
    row["target_artifact_id"]: row["source_evidence_unit_ids"]
    for row in ledger
    if row["projection_status"] == "projected"
}

## Native Local Search settings

GraphRAG 3.1.0 allocates its context with a 12000-token budget. The source-text and
community proportions are allocation settings, not fixed observed row counts.
Native entity matching and graph neighbourhood expansion determine the context;
the Sources list has no cosine score to reuse as a page relevance score.

In [3]:
display(
    Markdown(
        tabulate(
            [
                ("Community level", config.GRAPHRAG_COMMUNITY_LEVEL),
                ("Response type", config.GRAPHRAG_RESPONSE_TYPE),
                *config.GRAPHRAG_LOCAL_SEARCH.items(),
            ],
            headers=["Parameter", "Value"],
            tablefmt="github",
        )
    )
)

| Parameter                      | Value               |
|--------------------------------|---------------------|
| Community level                | 2                   |
| Response type                  | Multiple Paragraphs |
| max_context_tokens             | 12000               |
| text_unit_prop                 | 0.5                 |
| community_prop                 | 0.15                |
| conversation_history_max_turns | 5                   |
| top_k_entities                 | 10                  |
| top_k_relationships            | 10                  |

## From native Sources to source pages

`Sources.id` is a TextUnit human-readable ID. Its document ID identifies the
prepared input record; the input ledger supplies the original Evidence Unit.
Entity, relationship, report and claim rows are indirect context carriers.
The projection retains source order and invalid positions; it neither rescales
scores nor fills missing ranks. Identical content hashes cannot replace this mapping.

In [4]:
toy_context = {
    "sources": [{"id": 10}, {"id": 99}, {"id": 11}],
    "entities": [{"id": "E1"}],
}
toy_index = {"10": "INPUT-A", "11": "INPUT-B"}
toy_edges = {"INPUT-A": ["EU-A"], "INPUT-B": ["EU-B", "EU-C"]}
toy_records, toy_summary = project_local_search_context(
    "EXAMPLE", toy_context, toy_index, toy_edges
)
assert [row["rank"] for row in toy_records] == [1, 2, 3]
assert [row["projection_status"] for row in toy_records] == [
    "projected",
    "projection_failed",
    "ambiguous_carrier",
]
display(
    Markdown(
        tabulate(
            [
                [
                    row["rank"],
                    row["carrier_local_id"],
                    row["source_evidence_unit_id"],
                    row["projection_status"],
                ]
                for row in toy_records
            ],
            headers=["Rank", "Source ID", "Evidence Unit", "Projection"],
            tablefmt="github",
        )
    )
)

|   Rank |   Source ID | Evidence Unit   | Projection        |
|--------|-------------|-----------------|-------------------|
|      1 |          10 | EU-A            | projected         |
|      2 |          99 |                 | projection_failed |
|      3 |          11 |                 | ambiguous_carrier |

## Persist the native context without changing its order

Each native DataFrame retains its columns, index and ordered cell values. Missing
scalar values are represented explicitly. The following small conversion supports
the recorded native carrier tables; it does not rebuild or rank a new context.

In [5]:
def json_scalar(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if hasattr(value, "item") and not isinstance(
        value, (str, bytes, dict, list, tuple)
    ):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return {"special_float": str(value)}
    if isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_scalar(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_scalar(item) for item in value]
    if hasattr(value, "isoformat"):
        return value.isoformat()
    raise TypeError(f"Unsupported native context scalar: {type(value).__name__}")

In [6]:
def serialize_context(context_data: dict) -> dict:
    return {
        carrier: {
            "kind": "dataframe",
            "columns": list(table.columns),
            "index": [json_scalar(item) for item in table.index.tolist()],
            "data": [
                [json_scalar(value) for value in row]
                for row in table.itertuples(index=False, name=None)
            ],
        }
        for carrier, table in context_data.items()
    }


def context_records(record: dict) -> dict:
    return {
        carrier: [
            dict(zip(payload["columns"], row, strict=True)) for row in payload["data"]
        ]
        for carrier, payload in record["context_data"].items()
    }

## Execute the native Azure Local Search API

New generation loads the saved placeholder settings with environment values.
The direct library API receives the native graph tables, community level and response
type. Its own prompt and citation syntax remain unchanged. A failed or empty query
remains an error; it is not converted into a successful blank answer.

In [7]:
if RUN_AZURE_QUERY:
    assert data_dir.resolve().is_relative_to(Path("../local").resolve())
    configuration = json.loads(paths["configuration"].read_text(encoding="utf-8"))
    binding = {
        key: os.environ[key]
        for key in (
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_CHAT_DEPLOYMENT",
            embedding_variable,
        )
    }
    assert (
        hashlib.sha256(json.dumps(binding, sort_keys=True).encode("utf-8")).hexdigest()
        == configuration["azure_binding_sha256"]
    ), "Azure settings differ from the indexed run."
    assert configuration["settings"]["local_search"] == config.GRAPHRAG_LOCAL_SEARCH
    assert (
        configuration["settings"]["embedding_models"]["default_embedding_model"][
            "model"
        ]
        == condition["embedding_model"]
    )
    from graphrag.api.query import local_search as graphrag_local_search
    from graphrag.config.load_config import load_config

    with chdir(Path.cwd()):
        resolved = load_config(data_dir.resolve())
    native_path = data_dir / "native_results.jsonl"
    native_records = (
        [
            json.loads(line)
            for line in native_path.read_text(encoding="utf-8").splitlines()
            if line
        ]
        if native_path.exists()
        else []
    )
    assert len({row["question_id"] for row in native_records}) == len(native_records)
    for row in native_records:
        assert (
            row["index_manifest_sha256"] == index_sha
            and row["query_parameters"] == query_parameters
        )
        assert (
            row["question_id"] in question_texts
            and row["question_text"] == question_texts[row["question_id"]]
        )
        assert (
            row["status"] == "ok"
            and isinstance(row["response"], str)
            and row["response"].strip()
        )
        assert (
            hashlib.sha256(
                json.dumps(
                    row["context_data"], ensure_ascii=False, separators=(",", ":")
                ).encode("utf-8")
            ).hexdigest()
            == row["context_sha256"]
        )
    completed_ids = {row["question_id"] for row in native_records}

### Query the unanswered questions

Completed responses must match question text, index, query settings and context hash before reuse. The following loop calls the native API only for questions without a completed response. Each result is saved immediately.

In [8]:
if RUN_AZURE_QUERY:
    for question in questions:
        if question["question_id"] in completed_ids:
            continue
        response, native_context = await graphrag_local_search(
            config=resolved,
            entities=tables["entities"],
            communities=tables["communities"],
            community_reports=tables["community_reports"],
            text_units=tables["text_units"],
            relationships=tables["relationships"],
            covariates=None,
            community_level=config.GRAPHRAG_COMMUNITY_LEVEL,
            response_type=config.GRAPHRAG_RESPONSE_TYPE,
            query=question["question_text"],
        )
        assert isinstance(response, str) and response, (
            "Native Local Search returned no response."
        )
        serialized = serialize_context(native_context)
        record = {
            "question_id": question["question_id"],
            "question_text": question["question_text"],
            "status": "ok",
            "response": response,
            "context_data": serialized,
            "system_id": "graphrag",
            "run_id": index_manifest["run_id"],
            "condition_id": index_manifest["condition_id"],
            "question_set_version": index_manifest["question_set_version"],
            "query_parameters": query_parameters,
            "index_manifest_sha256": index_sha,
            "context_sha256": hashlib.sha256(
                json.dumps(
                    serialized, ensure_ascii=False, separators=(",", ":")
                ).encode("utf-8")
            ).hexdigest(),
        }
        with native_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        native_records.append(record)

## Read or project stored results

Only ordered Sources rows receive direct page-evidence credit. The mixed context
remains available for interpreting the original response. Missing and ambiguous
projection positions stay in the result, including when no valid nDCG can be computed.

In [9]:
if not RUN_AZURE_QUERY:
    results_manifest = json.loads(
        (data_dir / "results_manifest.json").read_text(encoding="utf-8")
    )
    result_paths = verify_artifacts(data_dir, results_manifest)
    native_records = [
        json.loads(line)
        for line in result_paths["native_results"]
        .read_text(encoding="utf-8")
        .splitlines()
        if line
    ]
    saved_projections = [
        json.loads(line)
        for line in result_paths["projection_results"]
        .read_text(encoding="utf-8")
        .splitlines()
        if line
    ]
assert len(native_records) == len(questions)
assert {row["question_id"] for row in native_records} == set(question_texts)
assert len({row["question_id"] for row in native_records}) == len(native_records)
projections = []
for native in native_records:
    assert (
        native["index_manifest_sha256"] == index_sha
        and native["query_parameters"] == query_parameters
    )
    assert native["question_text"] == question_texts[native["question_id"]]
    assert (
        native["status"] == "ok"
        and isinstance(native["response"], str)
        and native["response"].strip()
    )
    carriers = context_records(native)
    assert (
        hashlib.sha256(
            json.dumps(
                native["context_data"], ensure_ascii=False, separators=(",", ":")
            ).encode("utf-8")
        ).hexdigest()
        == native["context_sha256"]
    )
    records, summary = project_local_search_context(
        native["question_id"], carriers, text_unit_index, projection_edges
    )
    projections.append(
        {"question_id": native["question_id"], "records": records, "summary": summary}
    )
if not RUN_AZURE_QUERY:
    assert projections == saved_projections
print(
    {"native_responses": len(native_records), "projected_questions": len(projections)}
)

{'native_responses': 4, 'projected_questions': 4}


## Retain responses and projections

The results manifest binds the unchanged native response/context file and the
deterministic source projection. It also binds the exact index manifest used.
New responses remain separate from accepted human judgements in the evaluation station.

In [10]:
if RUN_AZURE_QUERY:
    assert {row["question_id"] for row in native_records} == {
        row["question_id"] for row in questions
    }
    verify_artifacts(data_dir, index_manifest)
    projection_path = data_dir / "projection_results.jsonl"
    projection_path.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in projections),
        encoding="utf-8",
    )
    paths_to_bind = {
        "native_results": native_path,
        "projection_results": projection_path,
        "index_manifest": data_dir / "index_manifest.json",
    }
    results_manifest = {
        "schema_version": "graphrag-local-search-artifacts-v1",
        "status": "completed",
        "graphrag_version": config.GRAPHRAG_VERSION,
        "question_set_version": index_manifest["question_set_version"],
        "artifacts": {
            role: {
                "path": path.relative_to(data_dir).as_posix(),
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            }
            for role, path in paths_to_bind.items()
        },
    }
    (data_dir / "results_manifest.json").write_text(
        json.dumps(results_manifest, indent=2) + "\n", encoding="utf-8"
    )

## Context carriers and original response

The carrier counts below describe the context returned for the selected question.
They are not evidence scores. Full carrier tables remain in the saved context file;
the display limits its preview rows to keep the notebook readable.

In [11]:
selected = native_records[0]
carriers = context_records(selected)
display(Markdown(f"**Question:** {selected['question_text']}"))
display(
    Markdown(
        tabulate(
            [(name, len(rows)) for name, rows in carriers.items()],
            headers=["Context carrier", "Rows"],
            tablefmt="github",
        )
    )
)
for carrier, rows in carriers.items():
    preview = json.dumps(rows[:3], ensure_ascii=False, indent=2)
    display(
        HTML(
            f"<details><summary>{escape(carrier)}: first three records</summary><pre style='white-space: pre-wrap'>{escape(preview)}</pre></details>"
        )
    )
display(Markdown(selected["response"]))

**Question:** Which numeric KEEPER request code stops all monkey business, which response codes may a SIMIAN use, and how can an attacker misuse that request?

| Context carrier   |   Rows |
|-------------------|--------|
| reports           |      1 |
| relationships     |      6 |
| claims            |      0 |
| entities          |     20 |
| sources           |     12 |

# KEEPER Protocol: Stopping Monkey Activity and Associated Security Risks

Within the Infinite Monkey Protocol Suite (IMPS), the KEEPER protocol is responsible for managing communications between the ZOO (controller) and SIMIAN (monkey interface). It uses numeric request and response codes to regulate and monitor the status and activities of monkeys connected via SIMIAN devices.

## Numeric KEEPER Request Code to Stop Monkey Business

The numeric KEEPER request code that stops all monkey business is **7**, labeled as **STOP**. This request instructs the SIMIAN to cease all activity related to its interfacing with the monkey, effectively halting the monkey's operations within the IMPS framework [Data: Entities (47); Sources (6)].

## SIMIAN Response Codes to the STOP Request

Upon receiving a STOP request, the SIMIAN is required to respond with one of the several valid response codes. These include:

- **NORESPONSE (code 4)**: Indicates the SIMIAN cannot verify the monkey’s status or is not responding.
- **ALIVE (code 5)**: The monkey is alive despite the STOP command.
- **DEAD (code 6)**: The monkey is dead.
- **GONE (code 2)**: The monkey or SIMIAN is no longer present or active [Data: Entities (50, 52, 54), Relationships (88, 108); Sources (6, 7)].

It is implementation-specific how exactly the SIMIAN halts the monkey, but it should aim to preserve the monkey’s ALIVE status to prevent external interventions such as shutdowns by authorities or animal rights groups [Data: Sources (6, 7)].

## Security Vulnerabilities: Misuse of the STOP Request

The STOP command introduces a significant **denial of service (DoS) vulnerability** within the IMPS network. An attacker could exploit this by sending a single STOP request to each SIMIAN device, thus disabling all monkeys' activity and causing widespread disruption in the network operations. This misuse can result in a massive denial of service throughout the entire ZOO, undermining the protocol's operational reliability [Data: Entities (62); Relationships (108); Sources (6, 16)].

Such vulnerabilities demand strong authentication and security measures to ensure that only authorized entities can issue STOP commands and to prevent malicious actors from exploiting this capability.

---

In summary, KEEPER request code 7 (STOP) halts all monkey actions, with SIMIAN devices permitted to respond using NORESPONSE, ALIVE, DEAD, or GONE codes. However, unauthorized STOP requests can cause denial of service attacks, highlighting critical protocol security concerns [Data: Entities (47, 50, 52, 54, 62), Relationships (88, 108), Sources (6, 7, 16)].

## Ranked source evidence

The first five source positions are evaluated with the same example qrels and
minimal evidence sets used by Classical RAG. Indirect graph carriers and unresolvable
source positions contribute no direct credit. Both systems use the same reference
pool; their native retrieval and answer mechanisms remain distinct.

In [12]:
if reference_path is not None:
    example = json.loads(reference_path.read_text(encoding="utf-8"))
    references = {
        q["question_id"]: q["reference_package"] for q in example["questions"]
    }
    assert example["question_set_version"] == index_manifest["question_set_version"]
    metric_rows = []
    for projection in projections:
        reference = references[projection["question_id"]]
        ranking = [row["source_evidence_unit_id"] for row in projection["records"]][
            : config.TOP_K
        ]
        score = ndcg_at_k(ranking, reference["qrels"])
        sets = [set(group) for group in reference["minimal_evidence_sets"]]
        complete = (
            any(group <= set(ranking) for group in sets)
            if sets and score is not None
            else None
        )
        page_ids = [unit.rsplit("::p", 1)[-1] if unit else None for unit in ranking]
        status = (
            "not applicable"
            if not reference["qrels"]
            else "valid"
            if score is not None
            else "invalid"
        )
        metric_rows.append(
            (projection["question_id"], page_ids, score, complete, status)
        )
    display(
        Markdown(
            tabulate(
                metric_rows,
                headers=[
                    "Question",
                    "Projected source pages",
                    "nDCG@5",
                    "Complete reference set",
                    "Status",
                ],
                tablefmt="github",
            )
        )
    )

| Question        | Projected source pages       |   nDCG@5 | Complete reference set   | Status         |
|-----------------|------------------------------|----------|--------------------------|----------------|
| DEMO-IMPS-Q-001 | ['7', '16', '10', '8', '17'] | 0.887915 | True                     | valid          |
| DEMO-IMPS-Q-002 | ['13', '12', '3', '4', '16'] | 1        | True                     | valid          |
| DEMO-IMPS-Q-003 | ['15', '4', '17', '1', '3']  | 1        | True                     | valid          |
| DEMO-IMPS-Q-004 | ['11', '4', '1', '10', '17'] |          |                          | not applicable |